# ADS-505 Team Project — DataCo Smart Supply Chain
## Initial EDA and Data Audit

**Working project direction:** Predicting late-delivery risk for proactive supply-chain intervention.

This notebook performs the shared initial EDA/data audit before modeling. It covers:
- dataset structure
- target distribution
- missing values
- duplicates
- unit of analysis
- target leakage
- numerical/categorical features
- initial feature-group ideas
- a simple baseline

The purpose is to understand and define the modeling dataset before building Logistic Regression, k-NN, or any later model.

In [ ]:
from pathlib import Path
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

In [ ]:
# Project folder on your computer
BASE_DIR = Path(r"C:\Users\bezaa\OneDrive\Desktop\USD\ADS 505\Project")

ZIP_PATH = BASE_DIR / "8gx2fvg2k6-5 (2).zip"
DATA_PATH = BASE_DIR / "DataCoSupplyChainDataset.csv"

print("Project folder:", BASE_DIR)

# Automatically extract the ZIP if the CSV is not already extracted.
if not DATA_PATH.exists():
    if ZIP_PATH.exists():
        print("Main CSV not found. Extracting ZIP file...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            zip_ref.extractall(BASE_DIR)
        print("Extraction complete.")
    else:
        raise FileNotFoundError(
            f"Could not find the dataset CSV or ZIP in:\n{BASE_DIR}\n\n"
            "Place 8gx2fvg2k6-5 (2).zip in that folder and run this cell again."
        )

print("Main dataset found:", DATA_PATH.exists())

In [ ]:
# Load DataCo main dataset
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

## 1. Dataset overview

In [ ]:
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_values": [df[col].nunique(dropna=True) for col in df.columns]
})

overview

In [ ]:
print(f"Raw rows / order-item records: {len(df):,}")
print(f"Unique orders: {df['Order Id'].nunique():,}")
print(f"Unique customers: {df['Customer Id'].nunique():,}")
print(f"Unique products: {df['Product Card Id'].nunique():,}")
print(f"Unique categories: {df['Category Id'].nunique():,}")
print(f"Unique departments: {df['Department Id'].nunique():,}")

rows_per_order = len(df) / df["Order Id"].nunique()
print(f"Average raw rows per order: {rows_per_order:.2f}")

In [ ]:
# Parse dates
df["order_date"] = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

df["shipping_date"] = pd.to_datetime(
    df["shipping date (DateOrders)"],
    errors="coerce"
)

print("Order date range:")
print(df["order_date"].min(), "to", df["order_date"].max())

print("\nShipping date range:")
print(df["shipping_date"].min(), "to", df["shipping_date"].max())

## 2. Target analysis

Candidate target: `Late_delivery_risk`

- `1` = late-delivery risk
- `0` = not late

In [ ]:
target_counts = df["Late_delivery_risk"].value_counts(dropna=False).sort_index()
target_pct = (
    df["Late_delivery_risk"]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_pct
})

target_summary

In [ ]:
ax = target_counts.plot(kind="bar", figsize=(7, 5))
ax.set_title("Late Delivery Risk Distribution")
ax.set_xlabel("Late_delivery_risk")
ax.set_ylabel("Number of records")
ax.set_xticklabels(["0 = Not late", "1 = Late"], rotation=0)
plt.tight_layout()
plt.show()

## 3. Missing-value audit

In [ ]:
missing = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100
    })
    .sort_values("missing_pct", ascending=False)
)

missing[missing["missing_count"] > 0].round(2)

In [ ]:
HIGH_MISSING_THRESHOLD = 50

high_missing = missing[
    missing["missing_pct"] >= HIGH_MISSING_THRESHOLD
].round(2)

high_missing

## 4. Duplicate audit

In [ ]:
exact_duplicates = df.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicates:,}")
print(f"Percent of dataset: {exact_duplicates / len(df) * 100:.2f}%")

In [ ]:
order_sizes = (
    df.groupby("Order Id")
    .size()
    .rename("rows_per_order")
)

print(order_sizes.describe())

ax = order_sizes.value_counts().sort_index().plot(kind="bar", figsize=(8, 5))
ax.set_title("Number of Order-Item Rows per Order")
ax.set_xlabel("Rows per Order")
ax.set_ylabel("Number of Orders")
plt.tight_layout()
plt.show()

## 5. Unit-of-analysis check

The raw dataset contains multiple rows for many orders. We need to decide whether the final modeling row should represent an **order item** or an **order**.

In [ ]:
target_values_per_order = (
    df.groupby("Order Id")["Late_delivery_risk"]
    .nunique(dropna=False)
)

conflicting_target_orders = target_values_per_order[
    target_values_per_order > 1
]

print(f"Orders with conflicting target values: {len(conflicting_target_orders):,}")

if len(conflicting_target_orders) == 0:
    print("Late_delivery_risk is consistent within each Order Id.")
else:
    display(conflicting_target_orders.head())

## 6. Target-leakage audit

For an early-warning model, we must exclude variables that reveal the outcome or are only known after the delivery result.

In [ ]:
delivery_status_target = pd.crosstab(
    df["Delivery Status"],
    df["Late_delivery_risk"],
    margins=True
)

delivery_status_target

In [ ]:
shipping_day_check = (
    df.groupby("Late_delivery_risk")[
        ["Days for shipping (real)", "Days for shipment (scheduled)"]
    ]
    .agg(["mean", "median", "min", "max"])
)

shipping_day_check

In [ ]:
df["actual_exceeds_scheduled"] = (
    df["Days for shipping (real)"] >
    df["Days for shipment (scheduled)"]
).astype(int)

pd.crosstab(
    df["actual_exceeds_scheduled"],
    df["Late_delivery_risk"],
    margins=True
)

### Leakage candidates

For the current early-warning framing, these should be excluded from predictors:

- `Delivery Status`
- `Days for shipping (real)`

Reason: they contain post-outcome information or directly reveal whether delivery was late.

## 7. Privacy, ID, and likely exclusion fields

In [ ]:
privacy_columns = [
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Password",
    "Customer Street",
    "Product Image"
]

id_columns = [
    "Customer Id",
    "Order Customer Id",
    "Order Id",
    "Order Item Id",
    "Order Item Cardprod Id",
    "Product Card Id",
    "Product Category Id",
    "Category Id",
    "Department Id"
]

leakage_candidates = [
    "Delivery Status",
    "Days for shipping (real)"
]

print("Privacy / identity fields:")
print(privacy_columns)

print("\nID fields to review:")
print(id_columns)

print("\nLeakage candidates:")
print(leakage_candidates)

## 8. Numerical-feature EDA

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

numeric_summary = df[numeric_cols].describe().T
numeric_summary["missing_pct"] = df[numeric_cols].isna().mean() * 100
numeric_summary["unique_values"] = df[numeric_cols].nunique()

numeric_summary.round(2)

In [ ]:
numeric_predictor_cols = [
    col for col in numeric_cols
    if col not in ["Late_delivery_risk", "actual_exceeds_scheduled"]
]

numeric_by_target = (
    df.groupby("Late_delivery_risk")[numeric_predictor_cols]
    .mean(numeric_only=True)
    .T
)

numeric_by_target.columns = ["Not_Late_Mean", "Late_Mean"]
numeric_by_target["Absolute_Difference"] = (
    numeric_by_target["Late_Mean"] -
    numeric_by_target["Not_Late_Mean"]
).abs()

numeric_by_target.sort_values(
    "Absolute_Difference",
    ascending=False
).round(3)

## 9. Categorical-feature EDA

In [ ]:
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

categorical_cardinality = pd.DataFrame({
    "column": categorical_cols,
    "unique_values": [df[c].nunique(dropna=True) for c in categorical_cols],
    "missing_pct": [df[c].isna().mean() * 100 for c in categorical_cols]
}).sort_values("unique_values", ascending=False)

categorical_cardinality.round(2)

In [ ]:
def late_rate_by_category(data, column, min_count=100, top_n=20):
    summary = (
        data.groupby(column, dropna=False)
        .agg(
            records=("Late_delivery_risk", "size"),
            late_rate=("Late_delivery_risk", "mean")
        )
        .reset_index()
    )

    summary = summary[summary["records"] >= min_count].copy()
    summary["late_rate_pct"] = summary["late_rate"] * 100

    return (
        summary.sort_values(
            ["records", "late_rate_pct"],
            ascending=[False, False]
        )
        .head(top_n)
    )

In [ ]:
shipping_mode_summary = late_rate_by_category(
    df, "Shipping Mode", min_count=1, top_n=20
)
shipping_mode_summary

In [ ]:
plot_data = shipping_mode_summary.sort_values("late_rate_pct")

ax = plot_data.plot(
    x="Shipping Mode",
    y="late_rate_pct",
    kind="bar",
    legend=False,
    figsize=(8, 5)
)

ax.set_title("Late-Delivery Rate by Shipping Mode")
ax.set_xlabel("Shipping Mode")
ax.set_ylabel("Late-Delivery Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
customer_segment_summary = late_rate_by_category(
    df, "Customer Segment", min_count=1, top_n=20
)
customer_segment_summary

In [ ]:
market_summary = late_rate_by_category(
    df, "Market", min_count=1, top_n=20
)
market_summary

In [ ]:
region_summary = late_rate_by_category(
    df, "Order Region", min_count=100, top_n=30
)
region_summary

In [ ]:
category_summary = late_rate_by_category(
    df, "Category Name", min_count=100, top_n=30
)
category_summary

## 10. Scheduled shipping days vs late-delivery risk

In [ ]:
scheduled_days_summary = (
    df.groupby("Days for shipment (scheduled)")
    .agg(
        records=("Late_delivery_risk", "size"),
        late_rate=("Late_delivery_risk", "mean")
    )
    .reset_index()
)

scheduled_days_summary["late_rate_pct"] = (
    scheduled_days_summary["late_rate"] * 100
)

scheduled_days_summary

In [ ]:
ax = scheduled_days_summary.plot(
    x="Days for shipment (scheduled)",
    y="late_rate_pct",
    kind="bar",
    legend=False,
    figsize=(8, 5)
)

ax.set_title("Late-Delivery Rate by Scheduled Shipping Days")
ax.set_xlabel("Scheduled Shipping Days")
ax.set_ylabel("Late-Delivery Rate (%)")
plt.tight_layout()
plt.show()

## 11. Date-based exploratory features

In [ ]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_dayofweek"] = df["order_date"].dt.day_name()
df["order_hour"] = df["order_date"].dt.hour

month_summary = (
    df.groupby("order_month")
    .agg(
        records=("Late_delivery_risk", "size"),
        late_rate=("Late_delivery_risk", "mean")
    )
    .reset_index()
)

month_summary["late_rate_pct"] = month_summary["late_rate"] * 100
month_summary

In [ ]:
ax = month_summary.plot(
    x="order_month",
    y="late_rate_pct",
    kind="line",
    marker="o",
    legend=False,
    figsize=(8, 5)
)

ax.set_title("Late-Delivery Rate by Order Month")
ax.set_xlabel("Month")
ax.set_ylabel("Late-Delivery Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_summary = (
    df.groupby("order_dayofweek")
    .agg(
        records=("Late_delivery_risk", "size"),
        late_rate=("Late_delivery_risk", "mean")
    )
    .reindex(weekday_order)
    .reset_index()
)

weekday_summary["late_rate_pct"] = weekday_summary["late_rate"] * 100
weekday_summary

## 12. High-correlation review

In [ ]:
corr_cols = [
    col for col in numeric_cols
    if col not in ["Late_delivery_risk", "actual_exceeds_scheduled"]
]

corr_matrix = df[corr_cols].corr(numeric_only=True)

pairs = []

for i, col_a in enumerate(corr_matrix.columns):
    for j, col_b in enumerate(corr_matrix.columns):
        if j <= i:
            continue

        corr_value = corr_matrix.loc[col_a, col_b]

        if abs(corr_value) >= 0.80:
            pairs.append({
                "feature_1": col_a,
                "feature_2": col_b,
                "correlation": corr_value
            })

high_corr_pairs = pd.DataFrame(
    pairs,
    columns=["feature_1", "feature_2", "correlation"]
)

if not high_corr_pairs.empty:
    high_corr_pairs = high_corr_pairs.sort_values(
        "correlation",
        key=np.abs,
        ascending=False
    )

high_corr_pairs

## 13. Initial column audit

In [ ]:
audit = pd.DataFrame({"column": df.columns})

audit["dtype"] = audit["column"].map(lambda c: str(df[c].dtype))
audit["missing_pct"] = audit["column"].map(
    lambda c: round(df[c].isna().mean() * 100, 2)
)
audit["unique_values"] = audit["column"].map(
    lambda c: df[c].nunique(dropna=True)
)

audit["initial_status"] = "REVIEW"
audit["reason"] = ""

audit.loc[
    audit["column"] == "Late_delivery_risk",
    ["initial_status", "reason"]
] = [
    "TARGET",
    "Candidate binary target for late-delivery prediction"
]

for c in leakage_candidates:
    if c in audit["column"].values:
        audit.loc[
            audit["column"] == c,
            ["initial_status", "reason"]
        ] = [
            "EXCLUDE - LEAKAGE",
            "Post-outcome information for an early-warning model"
        ]

for c in privacy_columns:
    if c in audit["column"].values:
        audit.loc[
            audit["column"] == c,
            ["initial_status", "reason"]
        ] = [
            "EXCLUDE - PRIVACY/IRRELEVANT",
            "Direct identity/privacy field or non-model image field"
        ]

for c in id_columns:
    if c in audit["column"].values:
        audit.loc[
            audit["column"] == c,
            ["initial_status", "reason"]
        ] = [
            "ID / REVIEW",
            "Identifier; do not use as an ordinary numeric predictor"
        ]

if "Product Description" in audit["column"].values:
    audit.loc[
        audit["column"] == "Product Description",
        ["initial_status", "reason"]
    ] = [
        "LIKELY EXCLUDE",
        "Very high / complete missingness"
    ]

for c in ["order date (DateOrders)", "shipping date (DateOrders)"]:
    if c in audit["column"].values:
        audit.loc[
            audit["column"] == c,
            ["initial_status", "reason"]
        ] = [
            "ENGINEER / REVIEW",
            "Create timing features only if available at the prediction point"
        ]

audit.sort_values(
    ["initial_status", "column"]
).reset_index(drop=True)

## 14. Preliminary feature-group ideas

Do not finalize these until you and Asher review the EDA.

### Group A — Order & Fulfillment
Possible examples:
- Shipping Mode
- Days for shipment (scheduled)
- order timing
- quantities
- order value
- discount/order complexity
- product/category mix

### Group B — Customer & Geography
Possible examples:
- Customer Segment
- Market
- Order Region
- Order Country
- destination/geographic characteristics

Both groups should eventually use the **same target, same cleaned dataset, same split, same models, and same metrics**.

## 15. Exploratory order-level prototype

In [ ]:
order_level = (
    df.groupby("Order Id")
    .agg(
        Late_delivery_risk=("Late_delivery_risk", "first"),
        Shipping_Mode=("Shipping Mode", "first"),
        Scheduled_Shipping_Days=("Days for shipment (scheduled)", "first"),
        Customer_Segment=("Customer Segment", "first"),
        Market=("Market", "first"),
        Order_Region=("Order Region", "first"),
        Order_Country=("Order Country", "first"),
        Order_Status=("Order Status", "first"),
        Order_Date=("order_date", "first"),
        Number_of_Order_Lines=("Order Item Id", "nunique"),
        Total_Quantity=("Order Item Quantity", "sum"),
        Total_Sales=("Sales", "sum"),
        Total_Order_Item_Value=("Order Item Total", "sum"),
        Mean_Discount=("Order Item Discount", "mean"),
        Mean_Discount_Rate=("Order Item Discount Rate", "mean"),
        Unique_Products=("Product Card Id", "nunique"),
        Unique_Categories=("Category Id", "nunique"),
        Unique_Departments=("Department Id", "nunique")
    )
    .reset_index()
)

print("Order-level prototype shape:", order_level.shape)
order_level.head()

In [ ]:
order_target_summary = pd.DataFrame({
    "count": order_level["Late_delivery_risk"].value_counts().sort_index(),
    "percent": (
        order_level["Late_delivery_risk"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

order_target_summary

## 16. Majority-class baseline

In [ ]:
def majority_baseline(target):
    counts = target.value_counts()
    majority_class = counts.idxmax()
    baseline_accuracy = counts.max() / counts.sum()
    return majority_class, baseline_accuracy

raw_majority_class, raw_baseline_accuracy = majority_baseline(
    df["Late_delivery_risk"]
)

order_majority_class, order_baseline_accuracy = majority_baseline(
    order_level["Late_delivery_risk"]
)

print("RAW ORDER-ITEM LEVEL")
print("Majority class:", raw_majority_class)
print(f"Baseline accuracy: {raw_baseline_accuracy:.4f}")

print("\nORDER LEVEL PROTOTYPE")
print("Majority class:", order_majority_class)
print(f"Baseline accuracy: {order_baseline_accuracy:.4f}")

## 17. Export EDA review tables

In [ ]:
EXPORT_DIR = BASE_DIR / "EDA_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

overview.to_csv(
    EXPORT_DIR / "dataset_column_overview.csv",
    index=False
)

missing.to_csv(
    EXPORT_DIR / "missing_value_summary.csv"
)

audit.to_csv(
    EXPORT_DIR / "initial_column_audit.csv",
    index=False
)

shipping_mode_summary.to_csv(
    EXPORT_DIR / "shipping_mode_late_rate.csv",
    index=False
)

customer_segment_summary.to_csv(
    EXPORT_DIR / "customer_segment_late_rate.csv",
    index=False
)

market_summary.to_csv(
    EXPORT_DIR / "market_late_rate.csv",
    index=False
)

region_summary.to_csv(
    EXPORT_DIR / "region_late_rate.csv",
    index=False
)

category_summary.to_csv(
    EXPORT_DIR / "category_late_rate.csv",
    index=False
)

high_corr_pairs.to_csv(
    EXPORT_DIR / "high_correlation_pairs.csv",
    index=False
)

print("EDA review files saved to:")
print(EXPORT_DIR)

# What to discuss with Asher after running this notebook

Write down answers to these questions:

1. What does one raw row represent?
2. Should the final modeling unit be **order-item** or **order**?
3. Is `Late_delivery_risk` still the best target?
4. Which columns clearly leak the outcome?
5. Which columns should be excluded because they are IDs, privacy fields, mostly missing, or irrelevant?
6. Which pre-outcome features show meaningful differences in late-delivery rate?
7. Which categorical variables have very high cardinality?
8. Which numerical variables are highly correlated/redundant?
9. What exact prediction point makes business sense?
10. What should the shared cleaned base dataset contain?
11. What is the majority-class baseline?
12. Which feature groups make the most sense after EDA?

**Do not start the final model yet.** First compare EDA findings and agree on the common base dataset, prediction point, leakage rules, and evaluation approach.